In [2]:
# 중첩 객체와 배열이 포함된 이커머스 영수증 스키마를 정의합니다.
receipt_schema = {
    "title": "ReceiptData",
    "description": "영수증 텍스트에서 결제 정보와 구매 품목 목록을 추출하는 스키마",
    "type": "object",
    "properties": {
        "store_info": {
            # 중첩 객체: 매장 정보
            "type": "object",
            "properties": {
                "store_name": {"type": "string", "description": "상점 이름"},
                "branch": {"type": "string", "description": "지점명"}
            },
            "required": ["store_name"]
        },
        "items": {
            # 배열 구조: 구매 품목 리스트
            "type": "array",
            "description": "구매한 상품 항목들의 목록",
            "items": {
                "type": "object",
                "properties": {
                    "item_name": {"type": "string", "description": "상품명"},
                    "quantity": {"type": "integer", "description": "구매 수량"},
                    "price": {"type": "integer", "description": "개당 가격"}
                },
                "required": ["item_name", "quantity", "price"]
            }
        },
        "payment_type": {
            "type": "string",
            "enum": ["카드", "현금", "간편결제"],
            "description": "결제 수단"
        }
    },
    "required": ["store_info", "items", "payment_type"]
}

# llm 교안 5-2 의 4.2
# 실습용 영수증 텍스트 데이터를 입력합니다.
receipt_text = """
[스마트스토어 강남점] 영수증
- 무선 마우스 2개 (개당 25,000원)
- C타입 케이블 1개 (개당 12,000원)
신용카드로 일시불 결제 완료되었습니다.
"""


In [3]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="gpt-6-luna",
    model_provider="openai",
    reasoning_effort="low"
)

structured_model = model.with_structured_output(receipt_schema)

result = structured_model.invoke(receipt_text)

print(result)

{'store_info': {'store_name': '스마트스토어', 'branch': '강남점'}, 'items': [{'item_name': '무선 마우스', 'quantity': 2, 'price': 25000}, {'item_name': 'C타입 케이블', 'quantity': 1, 'price': 12000}], 'payment_type': '카드'}


In [5]:
from pydantic import BaseModel, Field
from typing import List, Literal, Optional

# --- 중첩 데이터 모델 정의 ---

# 개별 구매 상품 정보를 담는 하위 모델을 정의합니다.
class PurchasedItem(BaseModel):
    # 상품명을 추출합니다.
    item_name: str = Field(description="상품 이름")
    # 구매 수량을 정수형으로 추출합니다.
    quantity: int = Field(description="구매 수량")
    # 개당 가격을 정수형으로 추출합니다.
    price: int = Field(description="개당 가격(원 단위)")

# 매장 정보를 담는 하위 모델을 정의합니다.
class StoreInfo(BaseModel):
    # 상점 이름을 추출합니다.
    store_name: str = Field(description="상점 이름")
    # 지점명을 추출하여 정보가 없으면 None을 허용합니다.
    branch: Optional[str] = Field(default=None, description="지점명")

# 전체 영수증 데이터를 통합하는 최상위 모델을 정의합니다.
class ReceiptAnalysis(BaseModel):
    # 매장 정보 모델을 중첩 객체로 포함합니다.
    store: StoreInfo = Field(description="매장 정보")
    # 여러 개의 품목을 리스트 형태로 수집합니다.
    items: List[PurchasedItem] = Field(description="구매한 상품 목록")
    # 결제 수단을 3가지 항목으로 엄격히 제한합니다.
    payment_type: Literal["카드", "현금", "간편결제"] = Field(description="결제 수단")


In [6]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model = "gpt-6-luna",
    model_provider="openai",
    reasoning_effort="low"
)

structured_model = model.with_structured_output(ReceiptAnalysis)

In [7]:
result = structured_model.invoke(receipt_text)

print(type(result))
print(result)

<class '__main__.ReceiptAnalysis'>
store=StoreInfo(store_name='스마트스토어', branch='강남점') items=[PurchasedItem(item_name='무선 마우스', quantity=2, price=25000), PurchasedItem(item_name='C타입 케이블', quantity=1, price=12000)] payment_type='카드'
